# Radar de Políticas Municipais — Análise Exploratória

**Notebook 01 — Exploração da primeira rodada de dados**

Este notebook explora os resultados da classificação de contratos do PNCP
por aderência à taxonomia de políticas municipais v0.2. A amostra coberta
aqui corresponde a contratos publicados em **15/09/2025** (1 dia útil),
limitada às primeiras 50 páginas da API (≈2.700 contratos brutos).

**Limitações conhecidas desta rodada:**
- Janela temporal muito curta (1 dia) — qualquer leitura agregada por
  município reflete picos contratuais, não política sustentada.
- A escala de "intensidade de documentação" não é proxy de implementação.
- A taxonomia v0.2 ainda admite falsos positivos sutis (ex: contratos de
  insumos genéricos para escolas de educação infantil contados em
  primeira_infancia/creche_pre_escola).

Estas limitações são propositais para esta etapa: o objetivo é validar a
metodologia end-to-end, não produzir um ranking definitivo.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'contratos_classificados.parquet')
agg = pd.read_parquet(ROOT / 'data' / 'processed' / 'municipios_eixos.parquet')
print(f'Contratos classificados: {len(df):,}')
print(f'Municípios únicos com pelo menos um registro: {df.codigo_ibge.nunique()}')
print(f'UFs representadas: {df.uf.nunique()}')

## 1. Distribuição por eixo

Quantos contratos foram classificados em cada eixo temático?

In [ ]:
dist_eixo = df.groupby('eixo_id').agg(
    n_classificacoes=('numero_controle_pncp', 'count'),
    n_contratos_unicos=('numero_controle_pncp', 'nunique'),
    n_municipios=('codigo_ibge', 'nunique'),
    valor_total=('valor_global', 'sum'),
).reset_index()
dist_eixo

## 2. Distribuição por subeixo

Mostra qual subeixo dominou cada eixo na amostra.

In [ ]:
df.groupby(['eixo_id', 'subeixo_id']).size().reset_index(name='n')

## 3. Top municípios por eixo

Atenção: ranking sobre 1 único dia útil. Tem valor diagnóstico (quem
está contratando algo naquele dia?) mas não substantivo (quem tem mais
política?). É exatamente este tipo de leitura enganosa que a
documentação metodológica precisa prevenir.

In [ ]:
for eixo in agg.eixo_id.unique():
    sub = agg[agg.eixo_id == eixo].sort_values('n_contratos', ascending=False).head(5)
    print(f'\n=== {eixo} ===')
    print(sub[['municipio','uf','n_contratos','valor_total','rotulo_nivel']].to_string(index=False))

## 4. Diagnóstico de qualidade — falsos positivos prováveis

Inspeção manual: contratos cujo objeto sugere classificação incorreta
ou borderline. Esta análise vira input pra próxima versão da taxonomia.

**Heurística simples:** contratos onde a única keyword acertada é uma
palavra curta e ambígua (`creche`, `caps`) sem keywords reforçadoras.

In [ ]:
# Contratos onde apenas 1 keyword acertou, e a keyword é uma das ambíguas
ambiguas = {'creche', 'caps', 'merenda'}
df['n_hits'] = df.keywords_hit.str.split(';').apply(len)
frag = df[(df.n_hits == 1) & (df.keywords_hit.isin(ambiguas))]
print(f'{len(frag)} contratos com sinal único e termo ambíguo:')
for _, r in frag.head(10).iterrows():
    print(f'  [{r.municipio}/{r.uf}] hit={r.keywords_hit}')
    print(f'    {(r.objeto or "")[:180]}')

## 5. Próximos passos sugeridos

1. **Expandir janela temporal** — repetir para 12 meses contínuos antes
   de tirar qualquer conclusão sobre intensidade municipal.
2. **Incluir Transferegov** — atravessa o gargalo do PNCP para municípios
   pequenos que contratam pouco mas recebem convênios.
3. **Score de confiança por contrato** — uma classificação com 1 keyword
   ambígua deve pesar menos que uma com 3 keywords + match de contexto.
4. **Validação assistida** — rotular manualmente uma amostra estratificada
   (n=300) para calcular precisão/recall por subeixo.